
# B3c — Panda on Downsampled Weather (Panda-side notebook)

**Purpose.** Evaluate the **published** Panda checkpoint (the same one
behind Experiment 8's and Experiment 10's advantage results — NOT
`baseline_100k`, which is confirmed undertrained relative to it) on
three conditions: native 10-minute Weather (H=96), hourly-downsampled
Weather at a fixed sample-horizon (H=96), and hourly-downsampled Weather
at a fixed physical-horizon (H=16, matching native's 16h span).

**Environment:** this notebook needs `transformers==4.40.2` (Panda's
pinned dependency) per the project's established two-environment
isolation pattern. Do **not** try to run Chronos in this same kernel —
use `b3c_chronos_downsampled_weather.ipynb` in a separate environment,
then bring both raw-prediction CSVs together in
`b3c_analysis_downsampled_weather.ipynb`.

**Model loading — verify against your harness before trusting results.**
The loading cell below uses the standard `PatchTSTPipeline.from_pretrained`
call, matching how the published checkpoint was loaded in earlier
advantage-based experiments (Experiment 8, 10, etc.), which is a
**different, simpler path** than the custom `load_patchtst_model` +
strict `state_dict` load required for your own `baseline_100k`/
`ablation_100k` checkpoints (those need the custom path because
`AutoModel(trust_remote_code=True)` silently resolves to stock
`transformers` PatchTST and breaks on `rmsnorm`). If this cell fails or
gives implausible results, swap in whichever exact loading cell you
used in `fixed_experiments.ipynb` for the published checkpoint — that
one is known-good and this is a best-guess reconstruction, not verified
against your actual working code in this session.



# B3c — Downsampled-Weather Model Intervention (shared window construction)

**This cell block must be identical, byte-for-byte, in both the Panda
notebook and the Chronos notebook.** The whole point of the design is
that both models see exactly the same context/target pairs, at exactly
the same real-world timestamps, at each resolution — otherwise the
advantage comparison isn't valid.

**Assumption flagged for verification:** this assumes `./ts_data/weather.csv`
(matching `fixed_experiments.ipynb`'s `DATA_DIR` convention) is the same
Jena/Max-Planck file used throughout this project (10-minute native
resolution, a `Date Time` column, and the same 21 numeric channels as
Experiment 8/30/31). Adjust `DATA_DIR` below if your path differs. If
your file uses a different datetime column name, edit `load_weather()`'s
`dt_col` detection before running.


In [1]:
import pandas as pd
import numpy as np

# ---- design constants (fixed before running, per the B3c design) ----
CONTEXT_LEN = 512            # samples, Panda's fixed native context length
N_WINDOWS = 20
NATIVE_H = 96                 # native (10-min) horizon -> 16h physical span
HOURLY_H_FIXED_SAMPLE = 96     # fixed-sample-horizon convention at hourly res (~4 days physical)
HOURLY_H_FIXED_PHYSICAL = 16   # fixed-physical-horizon convention at hourly res (16h, matches native)
DOWNSAMPLE_FACTOR = 6          # matches Experiment 31; hourly matches ETTh1's native sampling rate
SEED = 0

DATA_DIR = "./ts_data"   # matches fixed_experiments.ipynb's convention -- adjust if needed
WEATHER_PATH = f"{DATA_DIR}/weather.csv"


In [2]:
def load_weather(path=WEATHER_PATH):
    df = pd.read_csv(path)
    dt_col = 'Date Time' if 'Date Time' in df.columns else df.columns[0]
    sample = str(df[dt_col].dropna().iloc[0])
    if '.' in sample:
        df[dt_col] = pd.to_datetime(df[dt_col], dayfirst=True)   # raw Jena dd.mm.yyyy
    else:
        df[dt_col] = pd.to_datetime(df[dt_col])                  # ISO yyyy-mm-dd, unambiguous
    df = df.set_index(dt_col).sort_index()
    n_before = len(df)
    df = df[~df.index.duplicated(keep='first')]
    n_dropped = n_before - len(df)
    if n_dropped > 0:
        print(f'load_weather: dropped {n_dropped} duplicate-timestamp rows (kept first occurrence)')
    numeric_df = df.select_dtypes(include=[np.number])
    return numeric_df

def build_hourly(df_native):
    # Simple stride decimation (every DOWNSAMPLE_FACTOR-th native sample),
    # matching Experiment 31's downsampling method exactly -- NOT an hourly
    # average. This preserves point-sample character, consistent with how
    # ETTh1 itself is point-sampled rather than hour-averaged. If Experiment
    # 31's notebook used a different downsampling method (e.g. mean pooling),
    # switch this to match it exactly for comparability with that result.
    return df_native.iloc[::DOWNSAMPLE_FACTOR]

def valid_start_range(df_native, df_hourly):
    # Hourly context (512 hourly steps, ~21 days) spans far longer in real
    # time than native context (512 native steps, ~3.6 days), so hourly's
    # context requirement is the binding constraint on the left margin.
    # The right margin only needs to fit the larger of the three horizons
    # in physical time (hourly_H96_fixedsample = 4 days is the largest).
    min_start_time = df_hourly.index[CONTEXT_LEN]
    max_start_time = df_hourly.index[-1] - pd.Timedelta(hours=HOURLY_H_FIXED_SAMPLE)
    valid_native = df_native.loc[min_start_time:max_start_time]
    return valid_native.index

def get_window_start_timestamps(df_native, df_hourly, n_windows=N_WINDOWS):
    valid_idx = valid_start_range(df_native, df_hourly)
    positions = np.linspace(0, len(valid_idx) - 1, n_windows, dtype=int)
    return valid_idx[positions]

def make_window(df, start_ts, context_len, horizon_len):
    start_idx = df.index.get_indexer([start_ts], method='nearest')[0]
    if start_idx - context_len < 0 or start_idx + horizon_len > len(df):
        return None, None
    context = df.iloc[start_idx - context_len:start_idx]
    target = df.iloc[start_idx:start_idx + horizon_len]
    if len(context) < context_len or len(target) < horizon_len:
        return None, None
    return context.values.astype(np.float32), target.values.astype(np.float32)

def build_all_windows(weather_path=WEATHER_PATH):
    df_native = load_weather(weather_path)
    df_hourly = build_hourly(df_native)
    starts = get_window_start_timestamps(df_native, df_hourly)

    conditions = {
        'native_H96':             (df_native, CONTEXT_LEN, NATIVE_H),
        'hourly_H96_fixedsample': (df_hourly, CONTEXT_LEN, HOURLY_H_FIXED_SAMPLE),
        'hourly_H16_fixedphys':   (df_hourly, CONTEXT_LEN, HOURLY_H_FIXED_PHYSICAL),
    }

    windows = {name: [] for name in conditions}
    dropped = {name: 0 for name in conditions}
    for start_ts in starts:
        for name, (df, ctx_len, hor_len) in conditions.items():
            ctx, tgt = make_window(df, start_ts, ctx_len, hor_len)
            if ctx is None:
                dropped[name] += 1
                windows[name].append(None)
            else:
                windows[name].append((ctx, tgt))

    for name, n_dropped in dropped.items():
        if n_dropped > 0:
            print(f'WARNING: {name} dropped {n_dropped}/{N_WINDOWS} windows '
                  f'(insufficient context/horizon margin at that timestamp)')

    channels = list(df_native.columns)
    return windows, channels

windows, channels = build_all_windows()  # uses WEATHER_PATH = f"{DATA_DIR}/weather.csv" by default
print('Channels:', channels)
print('Conditions:', list(windows.keys()))
for name, wlist in windows.items():
    n_valid = sum(1 for w in wlist if w is not None)
    print(f'  {name}: {n_valid}/{N_WINDOWS} valid windows')


load_weather: dropped 1 duplicate-timestamp rows (kept first occurrence)
Channels: ['p (mbar)', 'T (degC)', 'Tpot (K)', 'Tdew (degC)', 'rh (%)', 'VPmax (mbar)', 'VPact (mbar)', 'VPdef (mbar)', 'sh (g/kg)', 'H2OC (mmol/mol)', 'rho (g/m**3)', 'wv (m/s)', 'max. wv (m/s)', 'wd (deg)', 'rain (mm)', 'raining (s)', 'SWDR (W/m�)', 'PAR (�mol/m�/s)', 'max. PAR (�mol/m�/s)', 'Tlog (degC)', 'OT']
Conditions: ['native_H96', 'hourly_H96_fixedsample', 'hourly_H16_fixedphys']
  native_H96: 20/20 valid windows
  hourly_H96_fixedsample: 20/20 valid windows
  hourly_H16_fixedphys: 20/20 valid windows


## Load the published Panda checkpoint


In [4]:
import torch
import sys
sys.path.insert(0, './panda')
from panda.patchtst.pipeline import PatchTSTPipeline

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {device}")

panda_pipeline = PatchTSTPipeline.from_pretrained(
    mode="predict",
    pretrain_path="GilpinLab/panda",
    device_map=device,
)
print("Loaded published Panda checkpoint.")

Device: cpu
Loaded published Panda checkpoint.


## Forecast helper

Panda's pipeline expects channel-first `(C, T)` input per the project's
established convention (`panda_mae_forecast` wrapper pattern from
earlier sessions). Context/target arrays from `build_all_windows` above
are `(T, C)` (time-major, matching the shared window-construction code),
so this wrapper transposes before calling `.predict()` and transposes
the output back.

In [10]:
context, target = windows['native_H96'][0]
mean = context.mean(axis=0, keepdims=True)
std = context.std(axis=0, keepdims=True) + 1e-8
context_norm = (context - mean) / std
context_ct = context_norm.T  # (C, T)
context_tensor = torch.tensor(context_ct, dtype=torch.float32)

print('context_tensor shape:', context_tensor.shape)

raw_pred = panda_pipeline.predict(context_tensor, prediction_length=NATIVE_H)
print('raw_pred type:', type(raw_pred))
print('raw_pred shape:', raw_pred.shape if hasattr(raw_pred, 'shape') else 'no .shape attr')

context_tensor shape: torch.Size([21, 512])
raw_pred type: <class 'torch.Tensor'>
raw_pred shape: torch.Size([1, 1, 128, 512])


In [11]:
import inspect
print(inspect.getsource(panda_pipeline.predict))
print(inspect.getsource(panda_pipeline._prepare_and_validate_context))

    @torch.no_grad()
    def predict(
        self,
        context: torch.Tensor | list[torch.Tensor],
        prediction_length: int,
        limit_prediction_length: bool = True,
        sliding_context: bool = False,
        verbose: bool = True,
    ) -> torch.Tensor:
        """
        Generate an autoregressive forecast for a given context timeseries

        Parameters
        ----------
        context
            Input series. This is either a 1D tensor, or a list
            of 1D tensors, or a 2D tensor whose first dimension
            is sequence length. In the latter case, use left-padding with
            ``torch.nan`` to align series of different lengths.
        prediction_length
            Time steps to predict. Defaults to what specified
            in ``self.model.config``.
        limit_prediction_length
            Force prediction length smaller or equal than the
            built-in prediction length from the model. True by
            default. When true, fai

In [19]:
def panda_forecast(context_tc, horizon):
    mean = context_tc.mean(axis=0, keepdims=True)
    std = context_tc.std(axis=0, keepdims=True)

    # Channels with ~zero variance in this window (e.g. rain=0 throughout a
    # dry period) would otherwise be divided by a near-zero std, producing
    # near-infinite normalized values that corrupt the joint prediction for
    # every other channel too (Panda attends over all channels jointly).
    # Guard: treat near-constant channels as already "normalized" (0-centered,
    # unscaled) rather than blowing them up.
    degenerate = std < 1e-6
    safe_std = np.where(degenerate, 1.0, std)
    context_norm = (context_tc - mean) / safe_std

    context_tensor = torch.tensor(context_norm, dtype=torch.float32)
    raw_pred = panda_pipeline.predict(context_tensor, prediction_length=horizon)
    pred = raw_pred.median(dim=1).values
    pred = pred[0, :horizon, :]
    pred_norm = pred.detach().cpu().numpy()

    return pred_norm, mean, safe_std, degenerate

## Run Panda on all three conditions, save raw predictions + per-window MAE

Per the project's raw-prediction-retention policy (Section 1.2, adopted
July 2026), both the raw forecasts and the per-window/per-channel MAE
are saved, so any future metric change is a re-scoring operation rather
than a rerun.

In [9]:
import traceback

context, target = windows['native_H96'][0]
try:
    pred = panda_forecast(context, NATIVE_H)
    print('OK, pred shape:', pred.shape)
except Exception:
    traceback.print_exc()

Traceback (most recent call last):
  File "C:\Users\user\AppData\Local\Temp\ipykernel_11724\3096366525.py", line 5, in <module>
    pred = panda_forecast(context, NATIVE_H)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\user\AppData\Local\Temp\ipykernel_11724\207183311.py", line 17, in panda_forecast
    pred_denorm = pred_tc * std + mean
                  ~~~~~~~~^~~~~
ValueError: operands could not be broadcast together with shapes (1,1,128,512) (1,21) 


In [13]:
context, target = windows['native_H96'][0]
pred = panda_forecast(context, NATIVE_H)
print('pred shape:', pred.shape, '-- should be (96, 21)')
print('target shape:', target.shape)

pred shape: (96, 21) -- should be (96, 21)
target shape: (96, 21)


In [17]:
per_pair = panda_df.sort_values('mae', ascending=False).head(15)
print(per_pair[['condition', 'channel', 'window_idx', 'mae']])

                  condition          channel  window_idx            mae
98               native_H96        rain (mm)           4  208333.328125
671  hourly_H96_fixedsample               OT          11      18.321764
33               native_H96    max. wv (m/s)           1       6.650602
32               native_H96         wv (m/s)           1       6.106091
182              native_H96        rain (mm)           8       4.528077
31               native_H96     rho (g/m**3)           1       3.526807
26               native_H96     VPmax (mbar)           1       3.290807
29               native_H96        sh (g/kg)           1       3.264858
30               native_H96  H2OC (mmol/mol)           1       3.256601
27               native_H96     VPact (mbar)           1       3.133526
28               native_H96     VPdef (mbar)           1       3.045789
21               native_H96         p (mbar)           1       3.029490
503  hourly_H96_fixedsample               OT           3       3

In [18]:
context, target = windows['native_H96'][1]
ctx_df = pd.DataFrame(context, columns=channels)
diag = pd.DataFrame({
    'min': ctx_df.min(), 'max': ctx_df.max(),
    'mean': ctx_df.mean(), 'std': ctx_df.std(),
})
print(diag.sort_values('std'))

                              min          max         mean         std
rain (mm)                0.000000     0.000000     0.000000    0.000000
sh (g/kg)                2.790000     3.980000     3.461250    0.311959
VPact (mbar)             4.500000     6.360000     5.558145    0.490032
H2OC (mmol/mol)          4.470000     6.380000     5.552852    0.499610
wv (m/s)                 0.190000     4.820000     1.636035    0.852379
VPdef (mbar)             0.660000     5.610000     2.122480    1.003304
max. wv (m/s)            0.440000     6.850000     2.744004    1.226123
Tdew (degC)             -4.110000     0.550000    -1.334824    1.236402
VPmax (mbar)             5.440000    11.430000     7.680449    1.348496
T (degC)                -1.590000     8.920000     2.995215    2.459868
Tlog (degC)              9.980000    20.500000    14.488241    2.597490
Tpot (K)               271.029999   282.429993   276.069702    2.632483
p (mbar)               993.549988  1007.729980  1001.110596    3

In [20]:
import os

os.makedirs('b3c_raw_predictions', exist_ok=True)
results = []
n_degenerate = 0

for condition, wlist in windows.items():
    horizon = {'native_H96': NATIVE_H,
               'hourly_H96_fixedsample': HOURLY_H_FIXED_SAMPLE,
               'hourly_H16_fixedphys': HOURLY_H_FIXED_PHYSICAL}[condition]
    for window_idx, w in enumerate(wlist):
        if w is None:
            continue
        context, target = w
        try:
            pred_norm, mean, safe_std, degenerate = panda_forecast(context, horizon)
        except Exception as e:
            print(f'FAILED: {condition} window {window_idx}: {type(e).__name__}: {e}')
            continue

        n_degenerate += degenerate.sum()
        target_norm = (target - mean) / safe_std

        np.savez(
            f'b3c_raw_predictions/panda_{condition}_w{window_idx:02d}.npz',
            context=context, target=target,
            forecast_norm=pred_norm, mean=mean, std=safe_std, degenerate=degenerate,
        )

        abs_err = np.abs(pred_norm - target_norm)
        for c_idx, channel_name in enumerate(channels):
            results.append({
                'model': 'panda', 'condition': condition, 'window_idx': window_idx,
                'channel': channel_name, 'mae': float(abs_err[:, c_idx].mean()),
                'degenerate': bool(degenerate[0, c_idx]),
            })

panda_df = pd.DataFrame(results)
panda_df.to_csv('b3c_panda_predictions.csv', index=False)
print(f'Degenerate (near-zero-std) channel-windows guarded: {n_degenerate}')
print(panda_df.groupby('condition')['mae'].agg(['mean', 'median', 'count']))

Degenerate (near-zero-std) channel-windows guarded: 8
                            mean    median  count
condition                                        
hourly_H16_fixedphys    0.496164  0.384531    420
hourly_H96_fixedsample  0.808017  0.696231    420
native_H96              0.765380  0.579580    420


## Sanity check against Experiment 8

The `native_H96` condition here should land close to Experiment 8's
Weather H=96 Panda MAE (0.6378 at n=20), as a check that this notebook's
harness/checkpoint matches the one that produced the original advantage
finding. A large mismatch here means the loading cell above needs
fixing before the downsampling result can be trusted at all.

In [21]:
native_mae = panda_df[panda_df.condition == 'native_H96']['mae'].mean()
print(f'This notebook, native_H96, mean Panda MAE across channels: {native_mae:.4f}')
print('Experiment 8 reference (Weather H=96, n=20): 0.6378')
print('If these differ substantially, verify the model-loading cell before trusting B3c.')


This notebook, native_H96, mean Panda MAE across channels: 0.7654
Experiment 8 reference (Weather H=96, n=20): 0.6378
If these differ substantially, verify the model-loading cell before trusting B3c.


In [22]:
df_native = load_weather()
df_hourly = build_hourly(df_native)

b3c_starts = get_window_start_timestamps(df_native, df_hourly)
print('B3c window start dates (hourly-constrained):')
print(pd.Series(b3c_starts).dt.date.tolist())
print('Span:', (b3c_starts.max() - b3c_starts.min()).days, 'days')

native_valid = df_native.index[CONTEXT_LEN : len(df_native) - NATIVE_H]
exp8_style_starts = native_valid[np.linspace(0, len(native_valid) - 1, N_WINDOWS, dtype=int)]
print('\nExp8-style window start dates (native-only constraint):')
print(pd.Series(exp8_style_starts).dt.date.tolist())
print('Span:', (exp8_style_starts.max() - exp8_style_starts.min()).days, 'days')
print('\nTotal dataset span:', (df_native.index.max() - df_native.index.min()).days, 'days')

load_weather: dropped 1 duplicate-timestamp rows (kept first occurrence)
B3c window start dates (hourly-constrained):
[datetime.date(2020, 1, 22), datetime.date(2020, 2, 9), datetime.date(2020, 2, 27), datetime.date(2020, 3, 16), datetime.date(2020, 4, 3), datetime.date(2020, 4, 20), datetime.date(2020, 5, 8), datetime.date(2020, 5, 26), datetime.date(2020, 6, 13), datetime.date(2020, 7, 1), datetime.date(2020, 7, 19), datetime.date(2020, 8, 6), datetime.date(2020, 8, 24), datetime.date(2020, 9, 11), datetime.date(2020, 9, 29), datetime.date(2020, 10, 17), datetime.date(2020, 11, 4), datetime.date(2020, 11, 22), datetime.date(2020, 12, 10), datetime.date(2020, 12, 27)]
Span: 340 days

Exp8-style window start dates (native-only constraint):
[datetime.date(2020, 1, 4), datetime.date(2020, 1, 23), datetime.date(2020, 2, 11), datetime.date(2020, 3, 1), datetime.date(2020, 3, 20), datetime.date(2020, 4, 8), datetime.date(2020, 4, 27), datetime.date(2020, 5, 16), datetime.date(2020, 6, 4), d

In [23]:
print(panda_pipeline.model.config.prediction_length)

128


In [24]:
df_native = load_weather()
native_valid = df_native.index[CONTEXT_LEN : len(df_native) - NATIVE_H]
exp8_style_starts = native_valid[np.linspace(0, len(native_valid) - 1, N_WINDOWS, dtype=int)]

exp8_mae = []
for start_ts in exp8_style_starts:
    ctx, tgt = make_window(df_native, start_ts, CONTEXT_LEN, NATIVE_H)
    if ctx is None:
        continue
    mean = ctx.mean(axis=0, keepdims=True)
    std = ctx.std(axis=0, keepdims=True)
    safe_std = np.where(std < 1e-6, 1.0, std)
    pred_norm, _, _, _ = panda_forecast(ctx, NATIVE_H)
    tgt_norm = (tgt - mean) / safe_std
    exp8_mae.append(np.abs(pred_norm - tgt_norm).mean())

print('Panda median MAE, Exp8-style windows:', np.median(exp8_mae))
print('Panda median MAE, B3c windows (already have): 0.7531')
print('Experiment 8 reference: 0.6378')

load_weather: dropped 1 duplicate-timestamp rows (kept first occurrence)
Panda median MAE, Exp8-style windows: 0.64397895
Panda median MAE, B3c windows (already have): 0.7531
Experiment 8 reference: 0.6378
